In [1]:
import numpy as np
import pandas as pd
import time
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

## Load V9 & Build Winning Features

In [5]:
train = pd.read_pickle('../data/processed/train_v9_engineered.pkl')
print(f'Loaded V9: {train.shape[0]:,} rows x {train.shape[1]} columns')
train['D1n'] = np.floor(train['TransactionDT'] / 86400) - train['D1']
for d in [4, 10, 15]:
    col = f'D{d}'
    if col in train.columns:
        train[f'D{d}n'] = np.floor(train['TransactionDT'] / 86400) - train[col]
train['uid_d1n'] = (train['card1'].astype(str) + '_' + train['addr1'].astype(str) + '_' + train['D1n'].astype(str))
grp = train.groupby('uid_d1n')
for col in ['D4n', 'D10n', 'D15n']:
    if col in train.columns:
        train[f'uid_{col}_mean'] = grp[col].transform('mean')
        train[f'uid_{col}_std']  = grp[col].transform('std')
if 'C13' in train.columns:
    train['uid_C13_nunique'] = grp['C13'].transform('nunique')
for col in ['V127', 'V136', 'V307', 'V309', 'V314', 'V320']:
    if col in train.columns:
        train[f'uid_{col}_nunique'] = grp[col].transform('nunique')
m_mapping = {'T': 1, 'F': 0, 'M0': 0, 'M1': 1, 'M2': 2}
for col in [f'M{i}' for i in range(1, 10)]:
    if col in train.columns:
        temp = train[col].map(m_mapping).astype(float).fillna(-1)
        train[f'uid_{col}_mean'] = temp.groupby(train['uid_d1n']).transform('mean')
for col in [f'C{i}' for i in range(1, 15) if i != 3]:
    if col in train.columns:
        train[f'uid_{col}_mean'] = grp[col].transform('mean')
if 'D9' in train.columns:
    train['uid_D9_mean'] = grp['D9'].transform('mean')
    train['uid_D9_std']  = grp['D9'].transform('std')
if 'P_emaildomain' in train.columns:
    train['uid_P_emaildomain_nunique'] = grp['P_emaildomain'].transform('nunique')
if 'id_02' in train.columns:
    train['uid_id_02_nunique'] = grp['id_02'].transform('nunique')

print(f'After FE: {train.shape[0]:,} rows x {train.shape[1]} columns')

new_cols = [c for c in train.columns if c.startswith('uid_') and c not in 
            ['uid', 'uid_transaction_count', 'uid_mean_amount', 'uid_std_amount',
             'uid_amount_ratio', 'uid_amount_zscore', 'uid_mean_hour']]
print(f'New winning features added: {len(new_cols)}')
print(new_cols)

Loaded V9: 590,540 rows x 442 columns
After FE: 590,540 rows x 486 columns
New winning features added: 40
['uid_d1n', 'uid_D4n_mean', 'uid_D4n_std', 'uid_D10n_mean', 'uid_D10n_std', 'uid_D15n_mean', 'uid_D15n_std', 'uid_C13_nunique', 'uid_V127_nunique', 'uid_V136_nunique', 'uid_V307_nunique', 'uid_V309_nunique', 'uid_V314_nunique', 'uid_V320_nunique', 'uid_M1_mean', 'uid_M2_mean', 'uid_M3_mean', 'uid_M4_mean', 'uid_M5_mean', 'uid_M6_mean', 'uid_M7_mean', 'uid_M8_mean', 'uid_M9_mean', 'uid_C1_mean', 'uid_C2_mean', 'uid_C4_mean', 'uid_C5_mean', 'uid_C6_mean', 'uid_C7_mean', 'uid_C8_mean', 'uid_C9_mean', 'uid_C10_mean', 'uid_C11_mean', 'uid_C12_mean', 'uid_C13_mean', 'uid_C14_mean', 'uid_D9_mean', 'uid_D9_std', 'uid_P_emaildomain_nunique', 'uid_id_02_nunique']


## Prepare Data

In [6]:
drop_cols = ['isFraud', 'TransactionID', 'uid', 'uid_d1n', 'trans_hour']
X = train.drop(columns=drop_cols)
y = train['isFraud']
print(f'Modeling features: {X.shape[1]} columns')
print(f'Target distribution: {y.value_counts().to_dict()}')

Modeling features: 481 columns
Target distribution: {0: 569877, 1: 20663}


In [7]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=42)
cat_cols = X_train.select_dtypes(include=['category', 'object']).columns.tolist()
print(f'Train: {X_train.shape[0]:,} rows | Test: {X_test.shape[0]:,} rows')
print(f'Categorical columns: {len(cat_cols)}')

Train: 472,432 rows | Test: 118,108 rows
Categorical columns: 31


## Threshold Search Utility

In [8]:
thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50]

def threshold_search(model_name, y_true, y_proba):
    rows, best_f1, best_thresh = [], 0.0, 0.25
    for t in thresholds:
        preds = (y_proba >= t).astype(int)
        r = recall_score(y_true, preds)
        p = precision_score(y_true, preds)
        f = f1_score(y_true, preds)
        rows.append({
            'Model': model_name, 'Threshold': t,
            'Recall': round(r, 4), 'Precision': round(p, 4), 'F1': round(f, 4)
        })
        if f > best_f1:
            best_f1, best_thresh = f, t
    auc = roc_auc_score(y_true, y_proba)
    return rows, best_f1, best_thresh, auc

all_results = []

## 1. LightGBM

In [9]:
print('Training LightGBM...')
t0 = time.time()

for col in cat_cols:
    X_train[col] = X_train[col].astype('category')
    X_test[col] = X_test[col].astype('category')

lgb_model = lgb.LGBMClassifier(n_estimators=1000, learning_rate=0.05, num_leaves=31, random_state=42, n_jobs=-1)
lgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)], categorical_feature=cat_cols,callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
lgb_proba = lgb_model.predict_proba(X_test)[:, 1]
lgb_rows, lgb_f1, lgb_t, lgb_auc = threshold_search('LightGBM', y_test, lgb_proba)
print(f"LightGBM | F1={lgb_f1:.4f} @ {lgb_t} | AUC={lgb_auc:.4f} | {time.time()-t0:.1f}s\n")
all_results.extend(lgb_rows)

Training LightGBM...
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 16530, number of negative: 455902
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.612574 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 49850
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 480
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034989 -> initscore=-3.317101
[LightGBM] [Info] Start training from score -3.317101
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1000]	valid_0's binary_logloss: 0.0440115
LightGBM | F1=0.8245 @ 0.25 | AUC=

## 2. XGBoost

In [10]:
print('Training XGBoost...')
t0 = time.time()

X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([X_train_xgb[col], X_test_xgb[col]], axis=0).astype(str)
    le.fit(combined)
    X_train_xgb[col] = le.transform(X_train_xgb[col].astype(str))
    X_test_xgb[col] = le.transform(X_test_xgb[col].astype(str))

# Fix inf/NaN from division features
for df in [X_train_xgb, X_test_xgb]:
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.fillna(-999, inplace=True)

xgb_model = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method='hist', random_state=42, n_jobs=-1, eval_metric='logloss')
xgb_model.fit(X_train_xgb, y_train, eval_set=[(X_test_xgb, y_test)], verbose=False)
xgb_proba = xgb_model.predict_proba(X_test_xgb)[:, 1]
xgb_rows, xgb_f1, xgb_t, xgb_auc = threshold_search('XGBoost', y_test, xgb_proba)
print(f"XGBoost  | F1={xgb_f1:.4f} @ {xgb_t} | AUC={xgb_auc:.4f} | {time.time()-t0:.1f}s\n")
all_results.extend(xgb_rows)

Training XGBoost...
XGBoost  | F1=0.8046 @ 0.25 | AUC=0.9729 | 507.0s



## 3. CatBoost (Winning Config)

- `depth=8` — deeper trees
- `grow_policy='Lossguide'` — asymmetric growth (LightGBM-style)
- `l2_leaf_reg=3` — leaf regularization
- `task_type='GPU'` — CUDA acceleration
- `iterations=2000` with `early_stopping_rounds=100`

In [11]:
print('Training CatBoost...')
t0 = time.time()

cat_features_idx = [X_train.columns.get_loc(c) for c in cat_cols]

# Fix NaN in categoricals for GPU
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()
for col in cat_cols:
    arr_train = np.array(X_train_cat[col], dtype=object)
    arr_test  = np.array(X_test_cat[col], dtype=object)
    arr_train[pd.isna(arr_train)] = 'missing'
    arr_test[pd.isna(arr_test)]   = 'missing'
    X_train_cat[col] = arr_train
    X_test_cat[col]  = arr_test

cat_model = CatBoostClassifier(iterations=2000, learning_rate=0.05, depth=8, l2_leaf_reg=3, random_seed=42, verbose=False, early_stopping_rounds=100, grow_policy='Lossguide', task_type='GPU', devices='0')
cat_model.fit(X_train_cat, y_train, cat_features=cat_features_idx, eval_set=(X_test_cat, y_test), verbose=False)
cat_proba = cat_model.predict_proba(X_test_cat)[:, 1]
cat_rows, cat_f1, cat_t, cat_auc = threshold_search('CatBoost', y_test, cat_proba)
print(f"CatBoost | F1={cat_f1:.4f} @ {cat_t} | AUC={cat_auc:.4f} | {time.time()-t0:.1f}s\n")
all_results.extend(cat_rows)

Training CatBoost...
CatBoost | F1=0.8429 @ 0.25 | AUC=0.9780 | 391.4s



## Summary

In [12]:
print("=" * 65)
print(f"{'Model':<10} {'Best F1':>8} {'Thresh':>8} {'AUC':>8}")
print("-" * 65)
print(f"{'LightGBM':<10} {lgb_f1:>8.4f} {lgb_t:>8.2f} {lgb_auc:>8.4f}")
print(f"{'XGBoost':<10} {xgb_f1:>8.4f} {xgb_t:>8.2f} {xgb_auc:>8.4f}")
print(f"{'CatBoost':<10} {cat_f1:>8.4f} {cat_t:>8.2f} {cat_auc:>8.4f}")
print("=" * 65)

Model       Best F1   Thresh      AUC
-----------------------------------------------------------------
LightGBM     0.8245     0.25   0.9759
XGBoost      0.8046     0.25   0.9729
CatBoost     0.8429     0.25   0.9780


## All Thresholds

In [13]:
df_results = pd.DataFrame(all_results)
print(df_results.to_string(index=False))

   Model  Threshold  Recall  Precision     F1
LightGBM       0.10  0.8529     0.6574 0.7425
LightGBM       0.15  0.8263     0.7709 0.7976
LightGBM       0.20  0.8011     0.8399 0.8201
LightGBM       0.25  0.7769     0.8783 0.8245
LightGBM       0.30  0.7525     0.9062 0.8222
LightGBM       0.40  0.7046     0.9381 0.8048
LightGBM       0.50  0.6555     0.9586 0.7786
 XGBoost       0.10  0.8420     0.6338 0.7232
 XGBoost       0.15  0.8118     0.7501 0.7797
 XGBoost       0.20  0.7835     0.8195 0.8011
 XGBoost       0.25  0.7554     0.8608 0.8046
 XGBoost       0.30  0.7341     0.8897 0.8045
 XGBoost       0.40  0.6830     0.9262 0.7862
 XGBoost       0.50  0.6356     0.9522 0.7623
CatBoost       0.10  0.8720     0.6911 0.7711
CatBoost       0.15  0.8456     0.7905 0.8172
CatBoost       0.20  0.8234     0.8505 0.8367
CatBoost       0.25  0.8026     0.8876 0.8429
CatBoost       0.30  0.7818     0.9107 0.8413
CatBoost       0.40  0.7447     0.9430 0.8322
CatBoost       0.50  0.7072     0.

In [14]:
train.to_pickle('../data/processed/train_v10_winning_fe.pkl', protocol=4)
print(f'Saved V10: {train.shape[0]:,} rows x {train.shape[1]} columns')

Saved V10: 590,540 rows x 486 columns
